In [4]:
import shap

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.base import clone

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [ ]:
def shap_feature_importance(model, X, max_display=20, plot=True):
    """
    Compute SHAP feature importance with effect direction.
    
    Returns:
        df_importance: DataFrame with columns:
           - mean_abs_shap: magnitude of importance
           - mean_shap: signed direction
           - direction: textual interpretation
    """
    
    # Pick the right explainer
    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X)
    except Exception:
        explainer = shap.KernelExplainer(model.predict, X.sample(50))
        shap_values = explainer.shap_values(X, nsamples=200)
    
    # For binary classification, shap returns list of 2 classes → take class 1
    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    shap_values = np.array(shap_values)

    # Compute summary statistics
    mean_abs = np.abs(shap_values).mean(axis=0)
    mean_signed = shap_values.mean(axis=0)
    
    df = pd.DataFrame({
        "feature": X.columns,
        "mean_abs_shap": mean_abs,
        "mean_shap": mean_signed
    })

    # Interpret direction
    def dir_label(v):
        if v > 0: 
            return "↑ increases prediction"
        elif v < 0: 
            return "↓ decreases prediction"
        else: 
            return "no directional effect"
    
    df["direction"] = df["mean_shap"].apply(dir_label)

    df = df.sort_values("mean_abs_shap", ascending=False)

    if plot:
        shap.summary_plot(shap_values, X, max_display=max_display)

    return df
